# Test Embedder V2 vs V1

Compare performance of the improved embedder (v2) against the original (v1).

- **v1**: Original CLIP-style, width=16, rot=15°, elastic=0.06
- **v2**: Triplet loss, width=24, rot=25°, elastic=0.10, flips, stronger aug

Expected: v2 macro ≈ 0.59+ (vs v1 = 0.557)

In [ ]:
import os
import sys
import numpy as np
from pathlib import Path

# Setup paths
DATA_ROOT = Path("/workspace/data/ehl")
WORK_DIR = Path("/workspace/out")
SHARED = Path("/shared-docker")
AMINE = SHARED / "amine"
MOHAMED = SHARED / "mohamed"

sys.path.insert(0, str(AMINE))
sys.path.insert(0, str(MOHAMED))

# Import from eval_harness
from eval_harness import (
    build_image_index, cached_volume, read_csv, mrr,
    SIMULATORS, GRID, N_VAL, SEED
)

print(f"Data: {DATA_ROOT}")
print(f"Grid: {GRID}, N_val: {N_VAL}, Seed: {SEED}")
print(f"Levels: {list(SIMULATORS.keys())}")

In [ ]:
# Import embedder builders
from learned_embedder import build as build_v1

import importlib.util
spec = importlib.util.spec_from_file_location(
    "learned_embedder_v2", MOHAMED / "learned_embedder_v2.py"
)
v2_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v2_module)
build_v2 = v2_module.build

print("✓ Loaded learned_embedder v1")
print("✓ Loaded learned_embedder v2")

In [ ]:
# Build index and load pairs
print("Building image index...")
index = build_image_index(DATA_ROOT)
print(f"✓ Found {len(index)} images")

# Load dataset1 pairs
pairs = read_csv(DATA_ROOT / "dataset1" / "train_pairs.csv")
print(f"✓ Loaded {len(pairs)} pairs")

# Split into train/val (using eval_harness seed/split)
rng = np.random.default_rng(SEED)
rng.shuffle(pairs)
val_pairs = pairs[:N_VAL]
train_pairs = pairs[N_VAL:]

print(f"  Train: {len(train_pairs)}, Val: {len(val_pairs)}")

In [ ]:
# Train v1 (original)
print("\n" + "="*60)
print("Training V1 (Original)")
print("="*60)

import time
t0 = time.time()
embed_v1 = build_v1(train_pairs, index, GRID, cached_volume)
t1 = time.time()
print(f"\n✓ V1 trained in {t1-t0:.0f}s")

In [ ]:
# Train v2 (improved)
print("\n" + "="*60)
print("Training V2 (Improved)")
print("="*60)

t0 = time.time()
embed_v2 = build_v2(train_pairs, index, GRID, cached_volume)
t1 = time.time()
print(f"\n✓ V2 trained in {t1-t0:.0f}s")

In [ ]:
# Evaluate on d1/d2/d3 proxies
def eval_embedder(embed_fn, name):
    """Evaluate embedder on all three difficulty levels."""
    print(f"\nEvaluating {name}...")
    results = {}
    rng = np.random.default_rng(SEED)
    
    # Load and cache val volumes
    t0 = time.time()
    q_vols = [cached_volume(p["query_id"], index[p["query_id"]], GRID) for p in val_pairs]
    g_vols = [cached_volume(p["target_id"], index[p["target_id"]], GRID) for p in val_pairs]
    print(f"  Loaded {len(val_pairs)} pairs ({time.time()-t0:.1f}s)")
    
    # Evaluate each level
    for level_name, simulator in SIMULATORS.items():
        t0 = time.time()
        # Apply simulator and embed
        lvl_rng = np.random.default_rng(SEED + hash(level_name) % 1000)
        q_sim = [simulator(v, lvl_rng) for v in q_vols]
        g_sim = [simulator(v, lvl_rng) for v in g_vols]
        
        q_emb = np.stack([embed_fn(v) for v in q_sim])
        g_emb = np.stack([embed_fn(v) for v in g_sim])
        
        # Compute MRR
        score = mrr(q_emb, g_emb, np.arange(len(val_pairs)))
        results[level_name] = score
        print(f"    {level_name}: {score:.4f} ({time.time()-t0:.1f}s)")
    
    macro = np.mean(list(results.values()))
    results["macro"] = macro
    print(f"  MACRO: {macro:.4f}")
    return results

print("Ready to evaluate on d1/d2/d3")

In [ ]:
# Run evaluations
results_v1 = eval_embedder(embed_v1, "V1 (Original)")

In [ ]:
results_v2 = eval_embedder(embed_v2, "V2 (Improved)")

In [ ]:
# Compare results
import pandas as pd

comparison = pd.DataFrame({
    "V1 (Original)": results_v1,
    "V2 (Improved)": results_v2,
})
comparison["Δ (V2-V1)"] = comparison["V2 (Improved)"] - comparison["V1 (Original)"]
comparison["%Δ"] = (comparison["Δ (V2-V1)"] / comparison["V1 (Original)"] * 100).round(1)

print("\n" + "="*70)
print("COMPARISON")
print("="*70)
print(comparison.to_string())

# Verdict
print("\n" + "="*70)
if results_v2["macro"] > results_v1["macro"]:
    improvement = (results_v2["macro"] - results_v1["macro"]) / results_v1["macro"] * 100
    print(f"✓ V2 WINS: +{improvement:.1f}% ({results_v1['macro']:.4f} → {results_v2['macro']:.4f})")
    print("  → Ready for Tier 2 (TTA + loss tuning)")
else:
    print(f"✗ V1 is better or tied")
    print(f"  V1 macro: {results_v1['macro']:.4f}")
    print(f"  V2 macro: {results_v2['macro']:.4f}")
    print("  → Try Tier 2 improvements instead")

## Results Summary

### If V2 Wins (Likely)
1. Commit v2 as your new baseline
2. Implement Tier 2:
   - **TTA (test-time aug)**: Encode query/target 8× with different augments, average
   - **Loss tuning**: Try focal loss or margin-based improvements
3. Submit best version to Kaggle public LB

### If V1 Stays Better
1. Keep v1 as baseline
2. Jump to Tier 2 with v1 (TTA often helps more than architecture changes)
3. Or try ensemble: (v1 + v2) / 2

### Next: TTA Implementation (Optional, if time permits)
```python
def embed_tta(vol, embed_fn, n_aug=8):
    """Average embeddings over augmented versions."""
    embeddings = []
    for _ in range(n_aug):
        aug_vol = augment(vol, cfg)  # random augmentation
        embeddings.append(embed_fn(aug_vol))
    return np.mean(embeddings, axis=0)
```

Expected TTA gain: +1–2% on d2/d3 (helps deformation robustness).